# Appendix 10.7: Citations

- [Lesson](#lesson)
- [Exercise](#exercise)
- [Example Playground](#example-playground)

## Setup

Run the following setup cell to load your API key and establish the `ask_with_citations` helper function.

In [ ]:
!pip install anthropic

import anthropic

%store -r API_KEY
%store -r MODEL_NAME

client = anthropic.Anthropic(api_key=API_KEY)

def ask_with_citations(question, documents):
    """documents: list of (title, text) tuples."""
    content = []
    for title, text in documents:
        content.append({
            "type": "document",
            "source": {"type": "text", "media_type": "text/plain", "data": text},
            "title": title,
            "citations": {"enabled": True},
        })
    content.append({"type": "text", "text": question})

    return client.messages.create(
        model=MODEL_NAME,
        max_tokens=1000,
        temperature=0.0,
        messages=[{"role": "user", "content": content}],
    )

---

## Lesson

In Chapter 8 (Avoiding Hallucinations), you learned to reduce hallucinations by manually asking Claude to **"first extract relevant quotes, then answer from them."** That's a real, useful technique — but it's still just an instruction Claude might not follow perfectly, and the "quotes" it gives you are plain text you'd have to fuzzy-match back against the source yourself to verify they're real.

**Citations** make this structural instead of instructional. You pass your source material as a `document` content block with citations enabled, and Claude's answer comes back as a sequence of text segments, where segments that draw on the source carry a `citations` field pointing at the *exact* passage they came from (which document, and where in it). You get grounded, checkable answers without having to prompt-engineer your way there, and without having to trust that a quote Claude gives you is real — the citation mechanism only attaches to text the model actually grounded in the source, by construction.

**Where this shows up in real systems:**
- A legal or compliance assistant answering questions about a contract, where every claim needs to be traceable to a specific clause for a human reviewer to check.
- A financial analysis tool (the kind of prompt you wrote in Chapter 9's Financial Services exercise) summarizing a prospectus or earnings report, where a wrong unsourced number is a real liability.
- A RAG pipeline over internal documentation, where users need to click through from an answer to the source page to trust it.
- **The 2 AM production bug this prevents:** a support bot summarizes a policy doc, states a refund window confidently, and it's subtly wrong because it blended two policy sections. With citations, that same wrong claim would either not be produced (because it wasn't actually grounded in one passage) or would carry a citation a human can spot-check in seconds — versus discovering the error only after a customer complaint.

**Version note:** citations is a purpose-built API feature (not just a prompting trick) that composes with the document content-block format also used for long-context and search workflows — see Appendix 10.3.

### Example — grounded answers over a policy document

In [ ]:
policy_doc = (
    "Shipping Policy\n"
    "Standard shipping takes 5-7 business days and is free on orders over $50.\n"
    "Expedited shipping takes 2-3 business days and costs a flat $12.99 regardless of order size.\n\n"
    "Return Policy\n"
    "Items may be returned within 30 days of delivery for a full refund, provided they are unused and in "
    "original packaging. Refunds are issued to the original payment method within 5-7 business days of us "
    "receiving the returned item."
)

response = ask_with_citations(
    "How long does expedited shipping take, and what's the refund window?",
    documents=[("Store Policies", policy_doc)],
)

for block in response.content:
    print(f"--- {block.type} ---")
    print(block.text)
    citations = getattr(block, "citations", None)
    if citations:
        for c in citations:
            print(f"    [cited: \"{c.cited_text}\" from '{c.document_title}']")
    print()

Each answer segment comes with the exact source sentence it's grounded in — you (or your UI) can render that as a footnote or a highlight, and a human reviewer can verify it in seconds instead of re-reading the whole document.

---

## Exercise

### Exercise 10.7.1 - Cite the Right Document

Below are two short documents. Ask a question whose answer draws from **both**, and confirm (by inspecting `document_title` on the citations) that Claude correctly attributes each part of its answer to the right source rather than blending them.

In [ ]:
doc_a = ("Product A Warranty\nProduct A carries a 1-year limited warranty covering manufacturing defects only.")
doc_b = ("Product B Warranty\nProduct B carries a 3-year limited warranty covering manufacturing defects and "
         "accidental damage.")

# Your question goes here — this is the only field you should change
QUESTION = "[Replace this text]"

response = ask_with_citations(QUESTION, documents=[("Product A", doc_a), ("Product B", doc_b)])

cited_titles = set()
for block in response.content:
    print(block.text)
    for c in getattr(block, "citations", None) or []:
        cited_titles.add(c.document_title)
        print(f"    [cited: '{c.document_title}']")

print("\n--------------------------- GRADING ---------------------------")
print("Cites both documents:", cited_titles == {"Product A", "Product B"})

❓ If you want a hint, run the cell below!

In [ ]:
from hints import exercise_10_7_1_hint; print(exercise_10_7_1_hint)

### Congrats!

**Recap:** citations turn "ask Claude to quote its sources" from a prompting technique into a structural API guarantee — pass source material as `document` blocks with citations enabled, and grounded claims come back with pointers to the exact passage they're based on.

**Interview-answer framing:** "For any answer that needs to be traceable to a source — legal, financial, support docs — I pass the source as a `document` content block with citations enabled rather than relying on a prompt asking Claude to quote things. The response comes back with structured citation objects pointing at the exact source passage behind each claim, which is both more reliable than manually-requested quotes and gives me something concrete to show a human reviewer or render in a UI."

Head over to Appendix 10.8 to learn about computer use and agents.

---

## Example Playground

This is an area for you to experiment freely with the prompt examples shown in this lesson.

In [ ]:
policy_doc = (
    "Shipping Policy\n"
    "Standard shipping takes 5-7 business days and is free on orders over $50.\n"
    "Expedited shipping takes 2-3 business days and costs a flat $12.99 regardless of order size.\n\n"
    "Return Policy\n"
    "Items may be returned within 30 days of delivery for a full refund, provided they are unused and in "
    "original packaging."
)

response = ask_with_citations(
    "How long does expedited shipping take, and what's the refund window?",
    documents=[("Store Policies", policy_doc)],
)

for block in response.content:
    print(block.text)
    for c in getattr(block, "citations", None) or []:
        print(f"    [cited: \"{c.cited_text}\"]")